# eph_05 — Temporal encoding: when does RT-predictive firing occur?

Fit ridge regression of log(RT) on spike counts across time bins spanning -5 to +2 s
relative to go cue. Identifies temporal windows where population firing best predicts
subsequent reaction time.

> **Code Ocean only:** requires `units_with_spikes` and session bundles (not available locally).
> Setup, function definitions, and visualization cells are importable locally but the fit loop is gated.

## 1. Setup

In [ ]:
%matplotlib inline
import contextlib, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "eph_05_temporal"
SAVE_FIG = False
print(f"ENV={ENV}  FOR_LOCAL={FOR_LOCAL}")

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter, filter_ephys_units, load_units_with_spike_times,
)
import pickle

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)
    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)
    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    filtered_ephys    = pd.read_pickle(FOR_LOCAL / "filtered_ephys.pkl")
    units_with_spikes = None
    base_dirs         = [FOR_LOCAL]
    print(f"Local dev: filtered_ephys {filtered_ephys.shape}")

## 3. Imports and time-bin config

In [ ]:
from typing import Sequence

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

from ephys_utils import AnalysisConfig, count_spikes_in_window, make_session_bundle
from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
analysis_window_s = (-5.0, 2.0)
bin_width_s = 0.25

bin_edges   = np.arange(analysis_window_s[0],
                         analysis_window_s[1] + 1e-9,
                         bin_width_s)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

print(f"{len(bin_centers)} bins, {bin_edges[0]*1000:.0f} to "
      f"{bin_edges[-1]*1000:.0f} ms, width={bin_width_s*1000:.0f} ms")

## 4. Analysis functions

`make_time_binned_unit_df` — counts spikes per bin per trial.
`fit_temporal_rt_model` — ridge regression log(RT) ~ bins.

In [ ]:
def make_time_binned_unit_df(
    unit_row: pd.Series,
    bundle: dict,
    bin_edges: Sequence[float],
    rt_col: str = "reaction_time_firstmove",
) -> pd.DataFrame:
    """Build a per-trial table with multi-bin spike counts + RT for one unit."""
    session = bundle["session"]
    unit_id = unit_row["unit_id"]

    spikes = np.asarray(unit_row["spike_times"], dtype=float) - bundle["session_offset"]
    spikes.sort()

    align_times   = bundle["align_times"]
    trial_features = bundle["trial_features"]
    n_bins = len(bin_edges) - 1

    records = []
    for tr, t0 in align_times.items():
        if not np.isfinite(t0):
            continue
        row = {"session": session, "unit_id": unit_id, "trial": int(tr)}
        for k in range(n_bins):
            row[f"spk_bin_{k}"] = int(count_spikes_in_window(
                spikes, t0, (bin_edges[k], bin_edges[k + 1])
            ))
        records.append(row)

    df_unit = pd.DataFrame(records).set_index("trial")
    df_unit = df_unit.join(trial_features, how="left").reset_index()
    return df_unit


def fit_temporal_rt_model(
    df_unit: pd.DataFrame,
    rt_col: str = "reaction_time_firstmove",
    prefix: str = "spk_bin_",
) -> dict:
    """Ridge regression: log(RT) ~ spike_counts across time bins. Returns beta per bin."""
    bin_cols = [c for c in df_unit.columns if c.startswith(prefix)]
    df = df_unit.dropna(subset=[rt_col] + bin_cols).copy()
    if len(df) < 10:
        raise ValueError("Insufficient trials.")
    X  = df[bin_cols].to_numpy(dtype=float)
    y  = np.log(df[rt_col].to_numpy(dtype=float))
    Xz = StandardScaler().fit_transform(X)
    model = RidgeCV(alphas=np.logspace(-3, 3, 20), store_cv_values=False)
    model.fit(Xz, y)
    return {
        "bin_cols": bin_cols,
        "beta":     model.coef_,
        "alpha":    model.alpha_,
        "intercept": model.intercept_,
    }

## 5. Fit loop

Iterates over all units, builds binned trial tables, fits `RidgeCV`, stores β vector.

In [ ]:
if ENV != "codeocean" or units_with_spikes is None:
    print("units_with_spikes not available locally — skipping fit loop.")
    print("On Code Ocean, this loop runs per unit and builds ridge_df.")
    ridge_df = None
else:
    cfg = AnalysisConfig(
        align_key="goCue",
        count_window_s=(0.0, 0.2),
        baseline_window_s=(-1.0, 0.0),
        min_trials_per_group=20,
    )
    bundle_cache = {}
    for sess in units_with_spikes["session"].unique():
        try:
            bundle_cache[sess] = make_session_bundle(sess, cfg, base_dirs)
        except Exception as e:
            print(f"  bundle failed: {sess}: {e}")

    results = []
    for u in units_with_spikes.itertuples(index=False):
        session = u.session
        if session not in bundle_cache:
            continue
        bundle   = bundle_cache[session]
        unit_row = pd.Series(u._asdict())
        try:
            df_binned = make_time_binned_unit_df(unit_row, bundle, bin_edges)
            bin_cols  = [c for c in df_binned.columns if c.startswith("spk_bin_")]
            df_clean  = df_binned.dropna(subset=["reaction_time_firstmove"] + bin_cols)
            if len(df_clean) < 10:
                continue
            fit = fit_temporal_rt_model(df_binned)
        except Exception as e:
            continue
        results.append({
            "session":  session,
            "unit_id":  unit_row.unit_id,
            "n_trials": len(df_clean),
            "alpha":    fit["alpha"],
            "beta":     fit["beta"],
        })

    ridge_df = pd.DataFrame(results)
    print(f"Fit {len(ridge_df)} units")

## 6. Per-unit β(t) curves

In [ ]:
if ridge_df is not None and len(ridge_df):
    fig, ax = plt.subplots(figsize=(8, 4))
    for _, row in ridge_df.iterrows():
        ax.plot(bin_centers * 1000, row["beta"], alpha=0.3, lw=0.6,
                color=PALETTE["neutral"])
    ax.axhline(0, ls="--", color="black", lw=1)
    ax.axvline(0,  ls=":",  color=PALETTE["accent"], lw=0.8, label="go cue")
    for be in bin_edges * 1000:
        ax.axvline(be, color="grey", ls=":", lw=0.5, alpha=0.4)
    ax.set_xlabel("Time after go cue (ms)")
    ax.set_ylabel("Ridge coefficient (log RT ~ spikes)")
    ax.set_title(f"Temporal RT sensitivity — {len(ridge_df)} units")
    ax.legend()
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "ridge_per_unit_curves", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("No ridge_df — run on Code Ocean to see per-unit curves.")

## 7. Population mean ± SEM β(t)

Negative β at pre-cue bins = units that fire more before go cue produce slower RT.
Negative β at early post-cue = units that fire more just after go produce slower RT.

In [ ]:
if ridge_df is not None and len(ridge_df):
    beta_mat  = np.stack(ridge_df["beta"].to_numpy(), axis=0)
    mean_beta = beta_mat.mean(axis=0)
    sem_beta  = beta_mat.std(axis=0, ddof=1) / np.sqrt(beta_mat.shape[0])
    bw_ms     = (bin_edges[1] - bin_edges[0]) * 1000.0

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, xlim, label in [
        (axes[0], None,        "Full window"),
        (axes[1], (-500, 1000), "Peri-cue"),
    ]:
        ax.errorbar(bin_centers * 1000, mean_beta, yerr=sem_beta,
                    xerr=bw_ms / 2, fmt="o-", capsize=0, color=PALETTE["neg"])
        ax.axhline(0, ls="--", color="black", lw=0.8)
        ax.axvline(0, ls=":",  color=PALETTE["accent"], lw=0.8, label="go cue")
        ax.set_xlabel("Time after go cue (ms)")
        ax.set_ylabel("Mean β (log RT ~ spikes)")
        ax.set_title(f"Population RT sensitivity — {label}")
        if xlim:
            ax.set_xlim(xlim)
        style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "ridge_mean_sem", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
else:
    print("No ridge_df available.")